# 2.11 Practical Application
## Expert-Level Capstone: Assignment, Object State, and Reproducible Quantitative Research

**Notebook:** `2.11_Practical_Application.ipynb`  
**Python:** 3.10+ · **Dependencies:** Python standard library only · **Level:** Expert / Quantitative Software Engineering

### Executive Overview

This final notebook integrates **all mechanisms in 2.1–2.10** into a single, auditable quantitative research workflow. An apparently innocuous assignment can either rebind a name, mutate a shared container, call a descriptor, consume an iterator, trigger an in-place method, or leave partially modified state after an exception. In a research system, these distinctions affect data provenance, point-in-time validity, realized P&L, accounting reconciliation, and reproducibility.

The capstone is a **Point-in-Time Research & Execution State Integrity Framework**. It manages immutable, versioned market observations; parses trade fills; validates type contracts and financial risk constraints; applies fills through copy-on-write state transitions; marks positions using only eligible historical quotations; exposes deterministic snapshots; detects stale state revisions; and reconstructs results by replaying committed fills.

**What this project demonstrates:** Python assignment semantics and a defensible financial state architecture—not a live execution system, exchange simulator, trading alpha model, or performance claim.

### Learning Outcomes

1. Differentiate names, identities, equality, mutable object graphs, and logical data versions.
2. Trace simple, chained, unpacking, attribute, subscription, and augmented assignments.
3. Explain the implications of right-hand-side evaluation and sequential target assignment.
4. Use annotations, immutable records, precise monetary types, and runtime validation together.
5. Eliminate silent aliasing between research trials and data revisions.
6. Diagnose closures, `nonlocal` bindings, walrus expressions, and context leakage.
7. Build immutable market data views with strict point-in-time lookups.
8. Apply trade fills using a single published immutable state reference.
9. Separate financial invariants from Python-level assignment behavior.
10. Verify rollback, replay equivalence, state-version conflicts, risk limits, and accounting identities.
11. Inspect risky assignments statically and explain what AST analysis cannot prove.
12. Create an extensive standard-library `unittest` verification suite.

### Critical Boundaries

- All amounts are **synthetic** and denominated in USD.
- `Decimal` is used for monetary and price calculations; values are not automatically rounded to cents at every intermediate step.
- The ledger uses **average-cost** accounting, long-only positions, no leverage, no borrowing, and no external cash flows.
- Market data represents synthetic *as-known* observations. Explicit revisions create new immutable market archives. This does **not** model all real vendor publication timestamps or survivorship-bias controls.
- Locks and single-reference publication protect **cooperating in-process callers**; there is no durable database, process-crash recovery, distributed consensus, or authentication.
- `assert` documents executable expectations. Financial input validation uses explicit exceptions because `assert` can be disabled with Python optimization.

**Execution guidance:** Restart the notebook kernel and run all cells in order. All intentional failure demonstrations catch their exceptions. The final tests are repeatable without a prior notebook run.

## Notebook Roadmap

| Section | Domain | Assignment mechanisms |
|---|---|---|
| 2.11.1 | Binding and execution-order trace | Names, attributes, subscriptions, RHS-first |
| 2.11.2 | Shared objects and defensive data | Identity, shallow/deep copy, read-only view |
| 2.11.3 | Non-atomic assignment failures | Target order, partial update, exceptions |
| 2.11.4 | Raw-fill parsing | Nested targets, starred unpacking, generators |
| 2.11.5 | In-place and pure accumulation | `+=`, `__iadd__`, `__add__`, aliasing |
| 2.11.6 | Type contracts | Annotations, `NewType`, Protocol, `Decimal` |
| 2.11.7 | Walrus expressions | One-pass evaluation, truthiness and missingness |
| 2.11.8 | Closure policies | Late binding, `nonlocal`, rebinding |
| 2.11.9 | Context isolation | `ContextVar`, tokens, separate contexts |
| 2.11.10 | Static inspection | `ast`, `symtable`, target analysis |
| 2.11.11 | Market archive | Immutable versioned quotes, point-in-time lookup |
| 2.11.12 | Financial records | Validated fills, positions, policy, snapshots |
| 2.11.13 | Pure financial transitions | Cash, average cost, P&L, invariant checks |
| 2.11.14 | Risk and valuation | Point-in-time marks, concentration, cash constraints |
| 2.11.15 | Transactional publication | Staging, locking, version checks, rollback |
| 2.11.16 | Reproducible demonstration | Deterministic numbers, revisions, failure paths |
| 2.11.17 | Replay and identity audit | Deterministic reconstruction, fingerprinting |
| 2.11.18 | Automated verification | Unit, negative, reconciliation, property tests |
| 2.11.19 | Final analytical interpretation | Semantics-to-engineering synthesis |
| 2.11.20 | Advanced exercises & challenge | Research and engineering extensions |

### Proof Obligations for the Capstone

For every accepted batch, the framework must demonstrate:

\[
\text{Cash}_{t}+\text{RemainingCostBasis}_{t}
=\text{InitialCash}+\text{RealizedPnL}_{t}.
\]

For every rejected batch:

\[
\text{LedgerState}_{\mathrm{after}}=\text{LedgerState}_{\mathrm{before}}.
\]

And for a frozen ledger state with marked-to-market NAV:

\[
\text{NAV}_t - \text{InitialCash}
=\text{RealizedPnL}_t+\text{UnrealizedPnL}_t.
\]

All identities assume the accounting model, no external cash flows, and consistent market marks. These are **economic accounting checks**, not evidence of predictive validity.

## 2.11.1 Language Semantics: Assignment Execution Trace

### Theoretical Foundations

Ordinary assignment evaluates the **right-hand-side expression before the assignment target**. A `Name` target rebinds a name; an `Attribute` target can invoke a setter; a `Subscript` target invokes item-assignment behavior. When an assignment has several targets, they are processed **left to right**; this does not make the operation transactional.

A useful experiment makes all observable steps explicit. Traces prove the behavior of the *specific* operations, while the language reference establishes the general rule. The order of bytecode instructions is an implementation detail and should not be hardcoded into tests.

In [1]:
events_211: list[str] = []

class TracedState:
    def __init__(self) -> None:
        object.__setattr__(self, "items", {})

    def __setattr__(self, key: str, value: object) -> None:
        events_211.append(f"attribute:{key}")
        object.__setattr__(self, key, value)

    def __setitem__(self, key: str, value: object) -> None:
        events_211.append(f"subscription:{key}")
        self.items[key] = value

traced = TracedState()

def rhs_value() -> int:
    events_211.append("rhs")
    return 25

def target_object() -> TracedState:
    events_211.append("target-object")
    return traced

def target_key() -> str:
    events_211.append("target-key")
    return "AAPL"

target_object()[target_key()] = rhs_value()
traced.risk_limit = rhs_value()

assert events_211 == [
    "rhs", "target-object", "target-key", "subscription:AAPL",
    "rhs", "attribute:risk_limit",
]
print("Execution trace:", events_211)
print("Stored values:", traced.items, traced.risk_limit)

Execution trace: ['rhs', 'target-object', 'target-key', 'subscription:AAPL', 'rhs', 'attribute:risk_limit']
Stored values: {'AAPL': 25} 25


### **Analytical Interpretation**

**1. Binding is not storage** — A name assignment updates a namespace binding; an attribute or item target may execute arbitrary user-defined methods.

**2. RHS-first semantics** — The first `rhs` event precedes the target-object and key evaluations.

**3. Target dispatch** — `__setitem__` controls the subscription, while `__setattr__` controls attribute assignment.

**4. Practical implication** — Trade-state update routines should avoid depending on hidden setter side effects; explicit transition functions are easier to validate.

## 2.11.2 Object Identity, Aliasing, Structural Sharing, and Snapshots

### Theoretical Foundations

`==` tests equality as implemented by objects; `is` tests whether two references identify the same object. `copy.copy` reconstructs the **outer** container while nested references may be shared. `copy.deepcopy` recursively copies supported graphs, preserving internal alias relationships through memoization; it is not a universal solution for resources, locks, or external objects.

`MappingProxyType` exposes a **live read-only view** of an underlying mapping. It does not make the original mapping immutable, and a change through another reference remains visible. A defensive snapshot instead creates independent immutable contents and does not retain a mutable data owner.

In [2]:
from copy import copy, deepcopy
from types import MappingProxyType

shared_quotes = {"AAPL": [100, 104], "MSFT": [200, 198]}
aliased_quotes = shared_quotes
shallow_quotes = copy(shared_quotes)
deep_quotes = deepcopy(shared_quotes)
read_only_live = MappingProxyType(shared_quotes)

assert aliased_quotes is shared_quotes
assert shallow_quotes is not shared_quotes
assert shallow_quotes["AAPL"] is shared_quotes["AAPL"]
assert deep_quotes["AAPL"] is not shared_quotes["AAPL"]

shared_quotes["AAPL"].append(106)
assert aliased_quotes["AAPL"] == [100, 104, 106]
assert shallow_quotes["AAPL"] == [100, 104, 106]
assert deep_quotes["AAPL"] == [100, 104]
assert read_only_live["AAPL"][-1] == 106

independent_snapshot = tuple(
    (ticker, tuple(values))
    for ticker, values in sorted(shared_quotes.items())
)
shared_quotes["AAPL"][-1] = 999
assert dict(independent_snapshot)["AAPL"][-1] == 106
print("Snapshot preserved:", independent_snapshot)

Snapshot preserved: (('AAPL', (100, 104, 106)), ('MSFT', (200, 198)))


### **Analytical Interpretation**

**1. Alias identity** — A second name pointing to a dictionary does not allocate an independent dictionary.

**2. Shallow copy** — The new mapping shares nested lists, so source-list mutation propagates.

**3. Deep copy** — The supported nested list structure is independent, although deep copying has cost and protocol limits.

**4. Live proxy vs snapshot** — A mapping proxy blocks writes through the proxy but still observes source changes. Immutable nested tuples captured in a new container preserve a historical state.

**5. Research relevance** — Versioned datasets must pin actual observation contents, not merely expose a read-only view of a still-mutable feed.

## 2.11.3 Partial Assignment, Overlapping Targets, and Exception Safety

### Theoretical Foundations

Right-hand-side evaluation preceding target assignment does **not** imply that the targets are updated simultaneously. A setter may reject a proposed value after an earlier target was already modified. Similarly, target evaluation can depend on a name rebound by an earlier target.

The safe alternative for multi-field financial records is **construct → validate → publish a complete new root state**, with no side effects during staging. Publication is one assignment to the managed state reference, within an explicit synchronization boundary. This is stronger than ad hoc rollback but does not provide durability or immunity to arbitrary external mutation.

In [3]:
class BoundedPolicy:
    def __init__(self) -> None:
        self._limit = 0.25
        self.positions = {"AAPL": 10}

    @property
    def limit(self) -> float:
        return self._limit

    @limit.setter
    def limit(self, value: float) -> None:
        if not 0 <= value <= 1:
            raise ValueError("Invalid limit")
        self._limit = value

unsafe = BoundedPolicy()
try:
    unsafe.positions["AAPL"], unsafe.limit = 20, 1.50
except ValueError:
    pass
else:
    raise AssertionError("Expected a validation failure")
assert unsafe.positions["AAPL"] == 20
assert unsafe.limit == 0.25

prices = [10, 20]
index = 0
index, prices[index] = 1, 99
assert (index, prices) == (1, [10, 99])

print("Partial update:", unsafe.positions, unsafe.limit)
print("Overlapping targets:", index, prices)

Partial update: {'AAPL': 20} 0.25
Overlapping targets: 1 [10, 99]


### **Analytical Interpretation**

**1. Partial update** — The first target changed before the property setter rejected the second. Exceptions do not reverse earlier mutations.

**2. Overlapping targets** — The subscription sees the newly bound value of `index`: assignment order changes the selected list element.

**3. Financial consistency** — Cash, holdings, and cost basis cannot safely be edited as unrelated fields if a later assignment might fail.

**4. Design response** — Publish an immutable aggregate state only after every intermediate financial invariant has been checked.

## 2.11.4 Extended Unpacking and One-Shot Trade Data

### Theoretical Foundations

A raw trade record may contain a fixed prefix plus a variable-length metadata tail. Starred unpacking is useful for isolating those fields; it constructs a list of the collected items. Iterable unpacking **consumes** its source, and an iterator may be exhausted or leave earlier targets assigned if a later operation fails. Never assume a generator can be read again.

For financial imports, parse raw strings into explicit domain objects *before* changing the ledger. Distinguish a missing metadata tail from a missing required transaction field. Source parsing must not rely on `eval`.

In [4]:
raw_trade_rows = (
    ("F001", "2026-01-02", "BUY", "AAPL", 100, "100.00", "2.00", "SIM", "batch-a"),
    ("F002", "2026-01-02", "BUY", "MSFT", 50, "200.00", "3.00", "SIM"),
    ("F003", "2026-01-05", "SELL", "AAPL", 20, "104.00", "1.00", "SIM", "batch-a"),
)

parsed_head, parsed_tail = [], []
for row in raw_trade_rows:
    fill_id, day, side, ticker, quantity, price, fee, *metadata = row
    parsed_head.append((fill_id, day, side, ticker, quantity, price, fee))
    parsed_tail.append(tuple(metadata))

assert len(parsed_head) == 3
assert parsed_tail == [("SIM", "batch-a"), ("SIM",), ("SIM", "batch-a")]
assert parsed_head[2][4] == 20

feed = iter([("AAPL", 100), ("MSFT", 50)])
a, b = feed
assert (a, b) == (("AAPL", 100), ("MSFT", 50))
assert list(feed) == []
print("Parsed records:", len(parsed_head), "metadata:", parsed_tail)

Parsed records: 3 metadata: [('SIM', 'batch-a'), ('SIM',), ('SIM', 'batch-a')]


### **Analytical Interpretation**

**1. Unpacking contracts** — Seven required fields are extracted, while the starred target captures any remaining metadata.

**2. No implicit replay** — A generator/iterator is consumed by unpacking; reading the same iterator later does not reproduce the original data.

**3. Input boundary** — Domain parsing, numerical validation, and immutable record construction should precede ledger mutation.

**4. Avoid silent data loss** — A real ingestion pipeline must additionally validate row schema, precision, duplicate IDs, and unexpected metadata according to a documented contract.

## 2.11.5 Augmented Assignment, Pure Addition, and Mutable Accumulators

### Theoretical Foundations

`target += value` can dispatch to `__iadd__`, mutate an existing object, and return it; when in-place behavior is unavailable, Python may use ordinary addition. `x + y` is not guaranteed to be pure for arbitrary user-defined objects, but a class can deliberately provide **pure `__add__`** and **mutating `__iadd__`** interfaces.

For shared mutable structures, different names may observe in-place updates. In an auditable research workflow, prefer an immutable aggregate whose addition explicitly returns a fresh state, or document an accumulator's ownership and mutation contract.

In [5]:
from dataclasses import dataclass
from decimal import Decimal

@dataclass(frozen=True, slots=True)
class ImmutablePnL:
    usd: Decimal

    def __add__(self, other: "ImmutablePnL") -> "ImmutablePnL":
        if not isinstance(other, ImmutablePnL):
            return NotImplemented
        return ImmutablePnL(self.usd + other.usd)

class MutablePnL:
    def __init__(self, usd: Decimal = Decimal("0")) -> None:
        self.usd = usd

    def __iadd__(self, other: ImmutablePnL) -> "MutablePnL":
        if not isinstance(other, ImmutablePnL):
            return NotImplemented
        self.usd += other.usd
        return self

original_pnl = ImmutablePnL(Decimal("10.00"))
scenario_pnl = original_pnl + ImmutablePnL(Decimal("2.50"))
assert original_pnl.usd == Decimal("10.00")
assert scenario_pnl.usd == Decimal("12.50")
assert original_pnl is not scenario_pnl

shared_pnl = MutablePnL(Decimal("10.00"))
alias_pnl = shared_pnl
shared_pnl += ImmutablePnL(Decimal("2.50"))
assert shared_pnl is alias_pnl
assert alias_pnl.usd == Decimal("12.50")
print("Independent scenario:", scenario_pnl.usd)
print("Shared mutable accumulator:", alias_pnl.usd)

Independent scenario: 12.50
Shared mutable accumulator: 12.50


### **Analytical Interpretation**

**1. Pure `+`** — The immutable record exposes a new value object while preserving the original scenario.

**2. In-place `+=`** — The mutable accumulator updates its own `usd` attribute and returns `self`.

**3. Aliasing consequence** — A second reference observes the same mutated accumulator.

**4. Design contract** — Pure state transitions and explicitly owned accumulators have different performance and audit trade-offs.

## 2.11.6 Annotated Assignments, Domain Types, and Runtime Validation

### Theoretical Foundations

Annotations document expected types and support static analysis; they **do not enforce runtime validation automatically**. `NewType` distinguishes semantically different string identifiers to type checkers while acting as an identity transformation at runtime. `Protocol` describes supported behavior structurally; `@runtime_checkable` protocols test only limited runtime attribute presence, not the full semantics of a financial interface.

A valid trading price is more restrictive than “numeric”: it must have an accepted representation, be finite, and be positive. We use `Decimal` throughout the capstone and explicitly reject binary floats at the import boundary to prevent accidental precision conversion.

In [6]:
from decimal import Decimal
from typing import NewType, Protocol, runtime_checkable, get_type_hints

FillId = NewType("FillId", str)
Ticker = NewType("Ticker", str)

@runtime_checkable
class HasSnapshot(Protocol):
    def snapshot(self) -> object: ...

annotated_cash: Decimal = Decimal("100000.00")
assert get_type_hints(__import__("__main__")).get("annotated_cash") in (None, Decimal)  # notebook-dependent module metadata
assert FillId("X-1") == "X-1"  # no runtime wrapper

def require_decimal(value: object, *, positive: bool = False) -> Decimal:
    if not isinstance(value, Decimal) or not value.is_finite():
        raise TypeError("Expected a finite Decimal")
    if positive and value <= 0:
        raise ValueError("Expected a positive Decimal")
    return value

assert require_decimal(Decimal("0.00")) == Decimal("0.00")
for invalid in (1.0, float("nan"), "10.00"):
    try:
        require_decimal(invalid)
    except TypeError:
        pass
    else:
        raise AssertionError("Invalid monetary representation was accepted")

print("Annotated amount:", annotated_cash)

Annotated amount: 100000.00


### **Analytical Interpretation**

**1. Annotations** — Type hints are metadata and can be used by external static type checkers, but ordinary assignment does not enforce them.

**2. Semantic identifiers** — `NewType` helps differentiate unrelated string domains statically but leaves runtime values as strings.

**3. Runtime contracts** — Explicit `Decimal` checks reject float and string inputs, avoiding silent coercion of externally supplied amounts.

**4. Precision qualification** — `Decimal` eliminates many binary float representation issues but still needs explicit quantization and rounding conventions in a production financial contract.

## 2.11.7 Walrus Operator, Truthiness, and Missing Values

### Theoretical Foundations

An assignment expression (`:=`) binds the evaluated value and returns it as part of a larger expression. Its strongest use cases avoid an expensive calculation being repeated and make a condition depend on the **actual computed value**, not an unrelated cached or global name.

A dangerous anti-pattern is `if (position := positions.get(ticker)):` when a valid empty or zero-valued result is possible. For financial state, **missing (`None`) is not equivalent to zero**, and checks should be explicit. In comprehensions, walrus bindings can target the containing scope; avoid silent name leakage in notebooks.

In [7]:
from decimal import Decimal

candidate_positions = {"AAPL": 0, "MSFT": 50}
lookup_calls: list[str] = []

def lookup_quantity(ticker: str) -> int | None:
    lookup_calls.append(ticker)
    return candidate_positions.get(ticker)

if (quantity := lookup_quantity("AAPL")) is not None:
    assert quantity == 0
else:
    raise AssertionError("Valid zero quantity was mistaken for missing")

if (missing := lookup_quantity("JPM")) is None:
    assert missing is None

assert lookup_calls == ["AAPL", "JPM"]
values = [Decimal("0"), Decimal("3"), Decimal("-2")]
nonzero = [result for x in values if (result := abs(x)) > 0]
assert nonzero == [Decimal("3"), Decimal("2")]
print("Zero vs missing:", quantity, missing, "nonzero:", nonzero)

Zero vs missing: 0 None nonzero: [Decimal('3'), Decimal('2')]


### **Analytical Interpretation**

**1. One-pass evaluation** — The assigned result can be reused without invoking the lookup twice.

**2. Missingness** — An explicit `is not None` check distinguishes a valid zero quantity from an absent position.

**3. Comprehension binding** — The walrus target is not just a temporary inside the comprehension: its binding can remain visible in the containing scope.

**4. Quant implication** — Confusing zero exposure with missing exposure can corrupt validation, universe inclusion, and risk-gate results.

## 2.11.8 Closures, Late Binding, and `nonlocal` Policy Versions

### Theoretical Foundations

Closures retain references to bindings in an enclosing scope; late binding means a callback may observe the final value of a loop variable rather than its iteration-specific value. Explicit factory arguments or carefully chosen default parameters can create independent per-trial bindings. `nonlocal` allows an inner function to *rebind* an existing enclosing binding.

In a research system, do not let parameter-search callbacks capture shared mutable loop state or a global risk configuration accidentally. A versioned closure can be useful for a small, controlled experimental interface, while immutable policy records are preferable for reproducibility.

In [8]:
from decimal import Decimal

thresholds = [Decimal("0.10"), Decimal("0.20"), Decimal("0.30")]
flawed_rules = [lambda x: x <= limit for limit in thresholds]
assert [rule(Decimal("0.15")) for rule in flawed_rules] == [True, True, True]

fixed_rules = [
    (lambda limit: (lambda x: x <= limit))(threshold)
    for threshold in thresholds
]
assert [rule(Decimal("0.15")) for rule in fixed_rules] == [False, True, True]

def policy_factory(initial: Decimal):
    current, version = initial, 0

    def read() -> tuple[Decimal, int]:
        return current, version

    def update(value: Decimal) -> None:
        nonlocal current, version
        require_decimal(value)
        if not Decimal("0") <= value <= Decimal("1"):
            raise ValueError("Invalid weight")
        current, version = value, version + 1

    return read, update

read_a, update_a = policy_factory(Decimal("0.20"))
read_b, update_b = policy_factory(Decimal("0.30"))
update_a(Decimal("0.25"))
assert read_a() == (Decimal("0.25"), 1)
assert read_b() == (Decimal("0.30"), 0)
print("Independent policy versions:", read_a(), read_b())

Independent policy versions: (Decimal('0.25'), 1) (Decimal('0.30'), 0)


### **Analytical Interpretation**

**1. Late-bound loop variable** — Every flawed callback resolves the same final `limit` binding after the loop finishes.

**2. Independent closure environments** — The factory produces one enclosing binding per threshold and per policy instance.

**3. Nonlocal state** — `nonlocal` allows controlled rebinding but does not make multiple statements transactional or thread-safe.

**4. Reproducibility** — Persist the exact policy values used for a run; a mutable policy closure alone is insufficient provenance.

## 2.11.9 Context Isolation for Independent Research Runs

### Theoretical Foundations

`ContextVar` provides context-local bindings useful for independent asynchronous tasks and explicit execution contexts. It is **not** the same as a process-global variable or thread-local mutable dictionary. `set()` returns a token that can restore the previous binding with `reset()`.

This example creates separate `Context` objects explicitly. The external context remains unchanged. A context variable whose value is a mutable object does not automatically freeze that object, so use immutable identifiers or immutable configuration values.

In [9]:
from contextvars import ContextVar, Context

active_research_id: ContextVar[str] = ContextVar("active_research_id", default="none")

def context_report(label: str) -> tuple[str, str]:
    token = active_research_id.set(label)
    try:
        return (label, active_research_id.get())
    finally:
        active_research_id.reset(token)

context_a, context_b = Context(), Context()
result_a = context_a.run(context_report, "run-A")
result_b = context_b.run(context_report, "run-B")
assert result_a == ("run-A", "run-A")
assert result_b == ("run-B", "run-B")
assert active_research_id.get() == "none"
print("Research context isolation:", result_a, result_b)

Research context isolation: ('run-A', 'run-A') ('run-B', 'run-B')


### **Analytical Interpretation**

**1. Binding is scoped to a context** — The two explicit contexts carry their own research run identifiers.

**2. Token-based restoration** — Resetting the token restores the prior binding even after an exceptional exit.

**3. Data ownership** — Context isolation protects bindings, not the internals of a mutable object shared between contexts.

**4. Finance use** — A provenance ID can follow one research task without introducing hidden cross-task global mutation.

## 2.11.10 Static Assignment Auditing with AST and Symbol Tables

### Theoretical Foundations

The `ast` module distinguishes `Name`, `Attribute`, and `Subscript` nodes in store context. For `x[i] = value`, the `Subscript` is a store target while its base name `x` is typically in load context. `symtable` reveals local, global, and free-variable classification but does **not** provide runtime alias analysis.

A static auditor can flag mutation-prone targets and augmented assignments for human review. It cannot prove that a setter has no side effects, that a financial invariant holds, or that an arbitrary program is safe to execute.

In [10]:
import ast
import symtable

assignment_source = "\n".join([
    "def update(book, positions, qty):",
    "    current = qty",
    "    positions['AAPL'] = current",
    "    book.cash = 100",
    "    current += 1",
    "    return current",
])
syntax_tree = ast.parse(assignment_source)
assignment_nodes = [
    node for node in ast.walk(syntax_tree)
    if isinstance(node, (ast.Assign, ast.AugAssign, ast.AnnAssign))
]
target_kinds = []
for node in assignment_nodes:
    targets = node.targets if isinstance(node, ast.Assign) else [node.target]
    target_kinds.extend(type(target).__name__ for target in targets)

assert sorted(target_kinds) == ["Attribute", "Name", "Name", "Subscript"]
symbols = symtable.symtable(assignment_source, "<audit>", "exec")
function_symbols = symbols.get_children()[0]
assert function_symbols.lookup("current").is_local()
assert function_symbols.lookup("positions").is_parameter()
print("Assignment target kinds:", sorted(target_kinds))
print("Function symbols:", [s.get_name() for s in function_symbols.get_symbols()])

Assignment target kinds: ['Attribute', 'Name', 'Name', 'Subscript']
Function symbols: ['book', 'positions', 'qty', 'current']


### **Analytical Interpretation**

**1. Store-context classification** — The AST identifies syntactic writes to names, attributes, and subscriptions.

**2. Symbol tables** — Compile-time symbol classification explains local and parameter bindings, not what values or objects are reached at runtime.

**3. Security and correctness limit** — Source inspection cannot guarantee freedom from aliasing, concurrent races, or financial calculation errors.

**4. Practical use** — Treat a static warning as a review signal to inspect ownership, mutation contracts, and tests.

## 2.11.11 Capstone: Immutable Market Archives and Point-in-Time Views

### Theoretical Foundations

A market-data correction should not mutate an old research result. An immutable archive stores independently constructed records, and a correction creates a **new archive version**. Each quote is associated with an observation day. `asof(day)` chooses, for each security, the most recent known observation **not later than the requested day**. This limits one common source of look-ahead leakage; it is not a complete point-in-time vendor-history model because real feeds also have publication times and historical revisions.

**Design invariants:** (1) archive inputs and nested records do not retain mutable containers; (2) `(ticker, observation_day)` is unique within a version; (3) revisions specify an expected old value; (4) revisions never rewrite the previous archive; (5) fingerprinting is deterministic for a canonical content encoding.

The monetary type contract is strict: a price must be a finite positive `Decimal`. `str(Decimal)` preserves its mathematical representation in the canonical audit payload; preserving exact text scale is not the same as preserving vendor feed provenance.

In [11]:
import hashlib
import json
from dataclasses import dataclass, replace
from datetime import date
from decimal import Decimal

def positive_decimal(value: object, label: str) -> Decimal:
    if not isinstance(value, Decimal) or not value.is_finite():
        raise TypeError(f"{label} must be a finite Decimal")
    if value <= 0:
        raise ValueError(f"{label} must be strictly positive")
    return value

def nonnegative_decimal(value: object, label: str) -> Decimal:
    if not isinstance(value, Decimal) or not value.is_finite():
        raise TypeError(f"{label} must be a finite Decimal")
    if value < 0:
        raise ValueError(f"{label} must be nonnegative")
    return value

@dataclass(frozen=True, slots=True)
class Quote:
    day: date
    ticker: str
    price_usd: Decimal

    def __post_init__(self) -> None:
        if type(self.day) is not date:
            raise TypeError("Quote day must be a date")
        if not isinstance(self.ticker, str) or not self.ticker.strip():
            raise ValueError("Ticker must be nonempty")
        positive_decimal(self.price_usd, "Price")

@dataclass(frozen=True, slots=True)
class MarketMark:
    archive_version: str
    asof: date
    quotes: tuple[Quote, ...]

    def price(self, ticker: str) -> Decimal:
        for quote in self.quotes:
            if quote.ticker == ticker:
                return quote.price_usd
        raise KeyError(f"No point-in-time price for {ticker!r}")

    def source_day(self, ticker: str) -> date:
        for quote in self.quotes:
            if quote.ticker == ticker:
                return quote.day
        raise KeyError(ticker)

@dataclass(frozen=True, slots=True)
class MarketArchive:
    version: str
    quotes: tuple[Quote, ...]

    def __post_init__(self) -> None:
        if not isinstance(self.version, str) or not self.version.strip():
            raise ValueError("Version must be nonempty")
        if not isinstance(self.quotes, (tuple, list)) or not self.quotes:
            raise ValueError("Archive requires quote records")
        if not all(isinstance(quote, Quote) for quote in self.quotes):
            raise TypeError("Archive elements must be Quote records")
        normalized = tuple(sorted(self.quotes, key=lambda q: (q.day, q.ticker)))
        keys = [(q.day, q.ticker) for q in normalized]
        if len(set(keys)) != len(keys):
            raise ValueError("Duplicate observation date and ticker")
        object.__setattr__(self, "quotes", normalized)

    def asof(self, day: date) -> MarketMark:
        if type(day) is not date:
            raise TypeError("asof requires a date")
        latest: dict[str, Quote] = {}
        for quote in self.quotes:
            if quote.day <= day:
                latest[quote.ticker] = quote
        if not latest:
            raise ValueError("No market data available as of date")
        return MarketMark(
            archive_version=self.version,
            asof=day,
            quotes=tuple(latest[t] for t in sorted(latest)),
        )

    def revise(
        self,
        *,
        day: date,
        ticker: str,
        expected_old_price: Decimal,
        revised_price: Decimal,
        new_version: str,
    ) -> "MarketArchive":
        if new_version == self.version:
            raise ValueError("Revision requires a new version identifier")
        positive_decimal(revised_price, "Revised price")
        matched = False
        rebuilt = []
        for quote in self.quotes:
            if quote.day == day and quote.ticker == ticker:
                if quote.price_usd != expected_old_price:
                    raise ValueError("Optimistic price precondition failed")
                rebuilt.append(replace(quote, price_usd=revised_price))
                matched = True
            else:
                rebuilt.append(quote)
        if not matched:
            raise KeyError("No quote matches the revision target")
        return MarketArchive(version=new_version, quotes=tuple(rebuilt))

    def fingerprint(self) -> str:
        payload = {
            "version": self.version,
            "quotes": [
                (q.day.isoformat(), q.ticker, str(q.price_usd))
                for q in self.quotes
            ],
        }
        wire = json.dumps(payload, separators=(",", ":"), sort_keys=True)
        return hashlib.sha256(wire.encode("utf-8")).hexdigest()

print("Market archive and as-of primitives defined.")

Market archive and as-of primitives defined.


### **Analytical Interpretation**

**1. Frozen object graph** — `Quote` records contain immutable dates, strings, and Decimals. `MarketArchive` constructs a sorted tuple and retains no client list.

**2. Point-in-time selection** — An as-of view excludes observations with later dates and returns the latest eligible quote for each ticker.

**3. Revision contract** — The caller must identify the expected old price; a mismatch rejects rather than silently overwrites.

**4. Hash semantics** — SHA-256 fingerprints verify the bytes of the canonical serialized representation, not the accuracy or authenticity of market prices.

**5. Limitations** — The model does not include exchange calendars, vendor ingestion timestamps, delisting, stock splits, or historical availability delays.

## 2.11.12 Capstone: Typed Fills, Immutable Positions, and Policy Contracts

### Theoretical Foundations

A **fill** is a historical event; it should not be silently modified after processing. A **position** is a value object consisting of shares and average acquisition cost. A **policy** describes explicit risk constraints used by every transition. A **ledger state** aggregates cash, realized P&L, positions, processed transaction IDs, fill history, archive provenance, and a version counter.

A frozen dataclass is **not automatically deeply immutable**. This model deliberately stores immutable tuples, `frozenset`, strings, and `Decimal`, and rejects invalid values at construction. The public state is still not an authorization or security boundary against hostile reflection.

**Accounting convention:** purchase fees are capitalized in average acquisition cost; sale fees reduce net proceeds and realized P&L. Each batch is one ledger *version* regardless of its number of fills.

In [12]:
from dataclasses import dataclass
from datetime import date
from decimal import Decimal
from typing import Literal

TradeSide = Literal["BUY", "SELL"]

@dataclass(frozen=True, slots=True)
class Fill:
    fill_id: str
    day: date
    side: TradeSide
    ticker: str
    shares: int
    execution_price_usd: Decimal
    fee_usd: Decimal

    def __post_init__(self) -> None:
        if not isinstance(self.fill_id, str) or not self.fill_id.strip():
            raise ValueError("fill_id must be nonempty")
        if type(self.day) is not date:
            raise TypeError("Fill day must be a date")
        if self.side not in ("BUY", "SELL"):
            raise ValueError("Unsupported trade side")
        if not isinstance(self.ticker, str) or not self.ticker.strip():
            raise ValueError("Ticker must be nonempty")
        if type(self.shares) is not int or self.shares <= 0:
            raise ValueError("Shares must be a positive int, excluding bool")
        positive_decimal(self.execution_price_usd, "Execution price")
        nonnegative_decimal(self.fee_usd, "Commission")

@dataclass(frozen=True, slots=True)
class Holding:
    shares: int
    avg_cost_usd: Decimal

    def __post_init__(self) -> None:
        if type(self.shares) is not int or self.shares <= 0:
            raise ValueError("Holding shares must be positive")
        positive_decimal(self.avg_cost_usd, "Average cost")

@dataclass(frozen=True, slots=True)
class RiskPolicy:
    max_position_weight: Decimal
    max_gross_weight: Decimal
    max_order_notional_usd: Decimal
    min_cash_usd: Decimal = Decimal("0")

    def __post_init__(self) -> None:
        p = positive_decimal(self.max_position_weight, "Maximum position weight")
        g = positive_decimal(self.max_gross_weight, "Maximum gross weight")
        positive_decimal(self.max_order_notional_usd, "Maximum order notional")
        nonnegative_decimal(self.min_cash_usd, "Minimum cash")
        if p > 1 or g > 1 or p > g:
            raise ValueError("Inconsistent concentration limits")

@dataclass(frozen=True, slots=True)
class LedgerState:
    version: int
    initial_cash_usd: Decimal
    cash_usd: Decimal
    realized_pnl_usd: Decimal
    holdings: tuple[tuple[str, Holding], ...]
    processed_ids: frozenset[str]
    fills: tuple[Fill, ...]
    archive_version: str | None

    def __post_init__(self) -> None:
        if type(self.version) is not int or self.version < 0:
            raise ValueError("Version must be a nonnegative integer")
        nonnegative_decimal(self.initial_cash_usd, "Initial cash")
        nonnegative_decimal(self.cash_usd, "Current cash")
        if not isinstance(self.realized_pnl_usd, Decimal) or not self.realized_pnl_usd.is_finite():
            raise TypeError("Realized P&L must be a finite Decimal")
        if not isinstance(self.holdings, tuple) or not isinstance(self.fills, tuple):
            raise TypeError("Ledger collections must be tuples")
        if not isinstance(self.processed_ids, frozenset):
            raise TypeError("Processed IDs must be a frozenset")
        tickers = [ticker for ticker, holding in self.holdings]
        if tickers != sorted(set(tickers)):
            raise ValueError("Holdings must be sorted and unique")
        if any(not isinstance(holding, Holding) for _, holding in self.holdings):
            raise TypeError("Holdings must contain Holding records")
        if any(not isinstance(fill, Fill) for fill in self.fills):
            raise TypeError("Fill history must contain Fill records")
        if frozenset(fill.fill_id for fill in self.fills) != self.processed_ids:
            raise ValueError("Processed IDs and fill history disagree")
        if len(self.fills) != len(self.processed_ids):
            raise ValueError("Duplicate fill IDs")
        # During staging, fills may exist before the batch version increments.
        if self.fills and (not self.archive_version):
            raise ValueError("Committed fills require a market archive version")

def parse_fill(row: tuple[object, ...]) -> Fill:
    """Parse a fixed prefix plus an optional metadata tail."""
    fill_id, raw_day, side, ticker, shares, raw_price, raw_fee, *metadata = row
    if not metadata or not all(isinstance(item, str) for item in metadata):
        raise ValueError("Expected nonempty string metadata")
    if type(raw_day) is not str or type(raw_price) is not str or type(raw_fee) is not str:
        raise TypeError("Date, price and fee fields must be strings")
    return Fill(
        fill_id=fill_id,
        day=date.fromisoformat(raw_day),
        side=side,
        ticker=ticker,
        shares=shares,
        execution_price_usd=Decimal(raw_price),
        fee_usd=Decimal(raw_fee),
    )

parsed_fills = tuple(parse_fill(row) for row in raw_trade_rows)
assert [fill.fill_id for fill in parsed_fills] == ["F001", "F002", "F003"]
print("Validated immutable trade fills:", len(parsed_fills))

Validated immutable trade fills: 3


### **Analytical Interpretation**

**1. Separate data contracts** — Trade events, holdings, risk constraints, and ledger states are different semantic types and cannot be silently substituted.

**2. Deeply stable public records** — All fields reachable through ordinary public state are immutable built-in or frozen value objects; source data lists are not retained.

**3. Runtime checks** — Annotations alone are insufficient; constructors enforce finite prices, valid share counts, and consistent transaction-ID history.

**4. Unpacking boundary** — Parsing turns an external positional record into an explicit typed immutable event before any portfolio state update.

**5. Security boundary** — Frozen dataclasses and in-process locks guard ordinary programming mistakes, not deliberately adversarial code or durable storage.

## 2.11.13 Capstone: Pure Transaction Transitions and Accounting Invariants

### Theoretical Foundations

A transition `next_state = apply_fill(old_state, fill, market, policy)` should be **pure with respect to the supplied objects**: it either constructs and returns a new immutable state, or raises a documented exception without mutating `old_state`. `dict(state.holdings)` creates a working mapping; holding records themselves remain immutable.

For a purchase, acquisition cost includes commission:

\[
C_{\mathrm{new}} = C_{\mathrm{old}} + qP + f,
\qquad
\bar P_{\mathrm{new}} = C_{\mathrm{new}}/(Q_{\mathrm{old}}+q).
\]

For a sale, cost basis of remaining shares is unchanged per share:

\[
\Delta\mathrm{RealizedPnL} = q(P-\bar P)-f.
\]

A post-transition accounting identity checks
\(\mathrm{Cash}+\mathrm{RemainingCostBasis}=\mathrm{InitialCash}+\mathrm{RealizedPnL}\).

**Important:** This algebra applies to the specified average-cost conventions, long-only inventory, and absence of cash transfers, taxes, and corporate actions. No externally supplied mark is needed to calculate realized P&L.

In [13]:
from dataclasses import replace
from decimal import Decimal

def remaining_cost_basis(state: LedgerState) -> Decimal:
    return sum(
        (holding.shares * holding.avg_cost_usd for _, holding in state.holdings),
        Decimal("0"),
    )

def assert_accounting_identity(state: LedgerState) -> None:
    left = state.cash_usd + remaining_cost_basis(state)
    right = state.initial_cash_usd + state.realized_pnl_usd
    # Decimal divisions can create repeating average costs.
    # Reconcile using an explicit monetary precision tolerance.
    tolerance = max(Decimal("1E-12"), max(abs(left), abs(right)) * Decimal("1E-24"))
    if abs(left - right) > tolerance:
        raise ArithmeticError(f"Ledger does not reconcile: {left} != {right}")

def initial_state(cash: Decimal) -> LedgerState:
    nonnegative_decimal(cash, "Initial cash")
    return LedgerState(
        version=0,
        initial_cash_usd=cash,
        cash_usd=cash,
        realized_pnl_usd=Decimal("0"),
        holdings=(),
        processed_ids=frozenset(),
        fills=(),
        archive_version=None,
    )

def transition_fill(
    state: LedgerState,
    fill: Fill,
    archive: MarketArchive,
    policy: RiskPolicy,
) -> LedgerState:
    """Apply one trade to a new immutable staged state."""
    if not isinstance(state, LedgerState) or not isinstance(fill, Fill):
        raise TypeError("Expected LedgerState and Fill")
    if not isinstance(archive, MarketArchive) or not isinstance(policy, RiskPolicy):
        raise TypeError("Expected MarketArchive and RiskPolicy")
    if fill.fill_id in state.processed_ids:
        raise ValueError("Duplicate fill ID")
    if state.archive_version not in (None, archive.version):
        raise ValueError("Existing fills are pinned to another archive")
    if state.fills and fill.day < state.fills[-1].day:
        raise ValueError("Fill dates must be nondecreasing")

    # Availability check without future quotation leakage.
    market_on_fill_day = archive.asof(fill.day)
    market_on_fill_day.price(fill.ticker)

    notional = fill.shares * fill.execution_price_usd
    if notional > policy.max_order_notional_usd:
        raise ValueError("Order notional exceeds risk limit")

    positions = dict(state.holdings)
    existing = positions.get(fill.ticker)
    old_qty = existing.shares if existing is not None else 0
    old_avg = existing.avg_cost_usd if existing is not None else Decimal("0")
    cash = state.cash_usd
    realized = state.realized_pnl_usd

    if fill.side == "BUY":
        outlay = notional + fill.fee_usd
        if cash - outlay < policy.min_cash_usd:
            raise ValueError("Insufficient cash")
        quantity = old_qty + fill.shares
        basis = old_qty * old_avg + outlay
        positions[fill.ticker] = Holding(quantity, basis / quantity)
        cash -= outlay
    else:
        if fill.shares > old_qty:
            raise ValueError("Insufficient shares")
        proceeds = notional - fill.fee_usd
        if proceeds < 0:
            raise ValueError("Selling fee exceeds proceeds")
        cash += proceeds
        realized += fill.shares * (fill.execution_price_usd - old_avg) - fill.fee_usd
        remaining = old_qty - fill.shares
        if remaining:
            positions[fill.ticker] = Holding(remaining, old_avg)
        else:
            del positions[fill.ticker]

    if not cash.is_finite() or not realized.is_finite():
        raise ArithmeticError("Nonfinite financial state")
    next_state = LedgerState(
        version=state.version,   # increment only at batch publication
        initial_cash_usd=state.initial_cash_usd,
        cash_usd=cash,
        realized_pnl_usd=realized,
        holdings=tuple(sorted(positions.items())),
        processed_ids=state.processed_ids | {fill.fill_id},
        fills=state.fills + (fill,),
        archive_version=archive.version,
    )
    assert_accounting_identity(next_state)
    return next_state

print("Pure financial transition function defined.")

Pure financial transition function defined.


### **Analytical Interpretation**

**1. Immutable input state** — `transition_fill` does not mutate `LedgerState` or its nested records; it constructs a new state object.

**2. Average-cost treatment** — Purchase fees increase position cost basis; sale fees reduce realized profit and net cash proceeds.

**3. Transaction chronology** — A fill cannot be processed out of chronological order and cannot use a later quotation for its market-availability check.

**4. Accounting invariant** — The cost-basis reconciliation checks a conservation relationship independent of mark-to-market quotations.

**5. Limitations** — The transition is deterministic under the recorded values but does not model order matching, slippage estimation, corporate actions, or real settlement rules.

## 2.11.14 Capstone: Point-in-Time Valuation and Risk Gates

### Theoretical Foundations

The mark-to-market view uses **market prices available as of the requested date** and an immutable ledger state. For long-only holdings:

\[
\mathrm{NAV}=\mathrm{Cash}+\sum_i Q_iP_i,
\qquad
\mathrm{UnrealizedPnL}=\sum_i Q_i(P_i-\bar P_i).
\]

Position concentration uses \(Q_iP_i/\mathrm{NAV}\), and gross long exposure uses \(\sum_i Q_iP_i/\mathrm{NAV}\). After **every staged fill**, the engine checks position and gross exposure constraints using the market view from the fill's own day; this prevents an apparently acceptable *final* portfolio from hiding a prohibited intermediate state.

This is a simplified deterministic research risk gate. It omits liquidity, volatility, market impact, sector constraints, and dynamic intraday mark changes.

In [14]:
from dataclasses import dataclass
from decimal import Decimal

@dataclass(frozen=True, slots=True)
class PortfolioValuation:
    asof: date
    archive_version: str
    nav_usd: Decimal
    market_value_usd: Decimal
    unrealized_pnl_usd: Decimal
    realized_pnl_usd: Decimal
    gross_weight: Decimal
    position_weights: tuple[tuple[str, Decimal], ...]

def mark_portfolio(state: LedgerState, mark: MarketMark) -> PortfolioValuation:
    if not isinstance(state, LedgerState) or not isinstance(mark, MarketMark):
        raise TypeError("Expected ledger state and market mark")
    market_values = tuple(
        (ticker, holding.shares * mark.price(ticker))
        for ticker, holding in state.holdings
    )
    exposure = sum((value for _, value in market_values), Decimal("0"))
    nav = state.cash_usd + exposure
    if not nav.is_finite() or nav <= 0:
        raise ValueError("NAV must be positive and finite")
    unrealized = exposure - remaining_cost_basis(state)
    weights = tuple((ticker, value / nav) for ticker, value in market_values)
    valuation = PortfolioValuation(
        asof=mark.asof,
        archive_version=mark.archive_version,
        nav_usd=nav,
        market_value_usd=exposure,
        unrealized_pnl_usd=unrealized,
        realized_pnl_usd=state.realized_pnl_usd,
        gross_weight=exposure / nav,
        position_weights=weights,
    )
    actual_change = nav - state.initial_cash_usd
    expected_change = state.realized_pnl_usd + unrealized
    tolerance = max(Decimal("1E-12"), max(abs(actual_change), abs(expected_change)) * Decimal("1E-24"))
    if abs(actual_change - expected_change) > tolerance:
        raise ArithmeticError("Mark-to-market P&L identity violated")
    return valuation

def enforce_risk(
    state: LedgerState,
    mark: MarketMark,
    policy: RiskPolicy,
) -> PortfolioValuation:
    if state.cash_usd < policy.min_cash_usd:
        raise ValueError("Cash reserve violated")
    valuation = mark_portfolio(state, mark)
    if valuation.gross_weight > policy.max_gross_weight:
        raise ValueError("Maximum gross exposure violated")
    for ticker, weight in valuation.position_weights:
        if weight > policy.max_position_weight:
            raise ValueError(f"Maximum position weight violated: {ticker}")
    return valuation

print("Point-in-time portfolio valuation and risk gate defined.")

Point-in-time portfolio valuation and risk gate defined.


### **Analytical Interpretation**

**1. Realized and unrealized components** — Realized P&L comes from processed sales; unrealized P&L compares market values with the remaining acquisition basis.

**2. NAV identity** — For this model, change in NAV equals realized plus unrealized P&L; no dividends or transfers are included.

**3. Point-in-time control** — The valuation uses the explicit selected market archive and as-of date; later quotes are excluded from the archive lookup.

**4. Concentration calculation** — Position and gross weights use contemporaneous marked market values divided by current NAV, not notional transaction values.

**5. Risk-boundary qualification** — These constraints support testable research policies but are not substitutes for a real pretrade risk system.

## 2.11.15 Capstone: One-Reference Commit and Optimistic State Versioning

### Theoretical Foundations

The managed ledger stores **one authoritative immutable `LedgerState` reference**. Each batch starts from that state, derives a succession of new temporary states, runs all financial and risk checks, then publishes a single completed state. No state is exposed during staging. A reentrant lock protects reads and writes by cooperating API callers.

`expected_version` is an **optimistic concurrency precondition**. It prevents a client from silently committing an update based on stale ledger state. Duplicate fill IDs are rejected. Empty batches fail. A ledger pins the market archive version used during its fills; applying later fills against a different archive version is explicitly rejected, while analytical repricing against a revised archive is allowed without modifying the ledger.

**Scope of guarantee:** exceptions raised during the pure transition/risk-staging path leave the authoritative state unchanged; public reads see an entire committed immutable state when callers use the API. This is not database ACID durability, authenticated mutation protection, or process-crash safety.

In [15]:
from dataclasses import replace
from threading import RLock

class ResearchLedger:
    """In-process immutable-root portfolio ledger with staged commits."""

    def __init__(self, *, cash_usd: Decimal, policy: RiskPolicy) -> None:
        if not isinstance(policy, RiskPolicy):
            raise TypeError("Expected a RiskPolicy")
        self._state = initial_state(cash_usd)
        self._policy = policy
        self._lock = RLock()

    def snapshot(self) -> LedgerState:
        with self._lock:
            return self._state  # safe to share: immutable domain graph

    @property
    def policy(self) -> RiskPolicy:
        return self._policy

    def commit_batch(
        self,
        fills: tuple[Fill, ...],
        *,
        archive: MarketArchive,
        expected_version: int,
        evaluation_day: date,
    ) -> LedgerState:
        if not isinstance(fills, tuple) or not fills:
            raise ValueError("Expected a nonempty tuple of fills")
        if type(expected_version) is not int or expected_version < 0:
            raise ValueError("Expected a nonnegative version")
        if type(evaluation_day) is not date:
            raise TypeError("Evaluation day must be a date")
        if not isinstance(archive, MarketArchive):
            raise TypeError("Expected a market archive")

        with self._lock:
            if expected_version != self._state.version:
                raise ValueError("Stale ledger version")
            if self._state.archive_version not in (None, archive.version):
                raise ValueError("Market archive version changed")

            staged = self._state
            for fill in fills:
                if not isinstance(fill, Fill):
                    raise TypeError("Batch contains a non-Fill record")
                if fill.day > evaluation_day:
                    raise ValueError("Fill occurs after evaluation day")
                staged = transition_fill(staged, fill, archive, self._policy)
                enforce_risk(staged, archive.asof(fill.day), self._policy)

            # Verify an additional valuation at the reporting date.
            enforce_risk(staged, archive.asof(evaluation_day), self._policy)
            completed = replace(staged, version=self._state.version + 1)

            # The single authoritative-root assignment occurs only here.
            self._state = completed
            return completed

print("Staged-commit portfolio ledger defined.")

Staged-commit portfolio ledger defined.


### **Analytical Interpretation**

**1. Root-state assignment** — The class replaces one `_state` reference rather than publishing a sequence of partly updated cash, holdings, and P&L attributes.

**2. Immutable staging** — Every fill generates a new `LedgerState`; failures abandon these temporary values without requiring a mutating rollback.

**3. Risk after every event** — The engine checks all post-trade staged holdings at the fill date and again at the reporting date.

**4. Optimistic versioning** — A stale `expected_version` is rejected before calculations or publication, protecting cooperating clients from lost-update behavior.

**5. Provenance pinning** — The ledger records its input archive version. A revised quote archive may reprice the same state but cannot silently rewrite its fill-processing source.

**6. Boundaries** — Locks, pure transitions, and immutable roots support in-process consistency; they do not supply database recovery or external event-delivery guarantees.

## 2.11.16 Deterministic End-to-End Research Scenario

### Theoretical Foundations

The synthetic archive contains daily close-like observations for AAPL, MSFT, and JPM. Three recorded fills represent two purchases and one partial AAPL sale. The ledger is evaluated as of **2026-01-05** using the original archive `market-v1`.

A revised market archive `market-v2` corrects the **2026-01-05 MSFT quote from $198 to $199**, without modifying the original archive or the committed portfolio state. This gives a controlled experiment in the difference between **changing the contents of a historical data version** and **rebinding an analysis to another immutable version**.

Expected baseline values:

| Metric | Expected |
|---|---:|
| Initial cash | $100,000.00 |
| Final cash after three fills | $82,074.00 |
| AAPL position | 80 shares, average cost $100.02 |
| MSFT position | 50 shares, average cost $200.06 |
| Realized P&L | +$78.60 |
| Baseline NAV (2026-01-05, v1) | $100,294.00 |
| Baseline unrealized P&L | +$215.40 |
| Revised NAV (2026-01-05, v2) | $100,344.00 |
| Mark-only revision impact | +$50.00 |
| Later NAV (2026-01-06, v1) | $100,804.00 |

The revised NAV is an **analytical sensitivity**, not a retroactive cash flow or an additional trade profit.

In [16]:
from datetime import date
from decimal import Decimal as D

def make_market_archive() -> MarketArchive:
    rows = (
        ("2026-01-02", "AAPL", "100"),
        ("2026-01-02", "MSFT", "200"),
        ("2026-01-02", "JPM", "50"),
        ("2026-01-05", "AAPL", "104"),
        ("2026-01-05", "MSFT", "198"),
        ("2026-01-05", "JPM", "52"),
        ("2026-01-06", "AAPL", "106"),
        ("2026-01-06", "MSFT", "205"),
        ("2026-01-06", "JPM", "51"),
    )
    return MarketArchive(
        version="market-v1",
        quotes=tuple(
            Quote(date.fromisoformat(day), ticker, D(price))
            for day, ticker, price in rows
        ),
    )

archive_v1 = make_market_archive()
archive_v2 = archive_v1.revise(
    day=date(2026, 1, 5),
    ticker="MSFT",
    expected_old_price=D("198"),
    revised_price=D("199"),
    new_version="market-v2",
)

policy = RiskPolicy(
    max_position_weight=D("0.20"),
    max_gross_weight=D("0.35"),
    max_order_notional_usd=D("20000"),
    min_cash_usd=D("0"),
)
research_ledger = ResearchLedger(cash_usd=D("100000"), policy=policy)
original_snapshot = research_ledger.snapshot()
completed_state = research_ledger.commit_batch(
    parsed_fills,
    archive=archive_v1,
    expected_version=0,
    evaluation_day=date(2026, 1, 5),
)
baseline = mark_portfolio(completed_state, archive_v1.asof(date(2026, 1, 5)))
revised = mark_portfolio(completed_state, archive_v2.asof(date(2026, 1, 5)))
later_mark = mark_portfolio(completed_state, archive_v1.asof(date(2026, 1, 6)))

assert completed_state.version == 1
assert completed_state.cash_usd == D("82074")
assert completed_state.realized_pnl_usd == D("78.60")
assert dict(completed_state.holdings)["AAPL"] == Holding(80, D("100.02"))
assert dict(completed_state.holdings)["MSFT"] == Holding(50, D("200.06"))
assert baseline.nav_usd == D("100294")
assert baseline.unrealized_pnl_usd == D("215.40")
assert revised.nav_usd == D("100344")
assert later_mark.nav_usd == D("100804")
assert archive_v1.asof(date(2026, 1, 5)).price("MSFT") == D("198")
assert original_snapshot.cash_usd == D("100000")

print("=" * 68)
print("QUANTITATIVE RESEARCH — STATE INTEGRITY REPORT")
print("=" * 68)
print("State version:", completed_state.version)
print("Pinned execution archive:", completed_state.archive_version)
print("Cash:", completed_state.cash_usd)
print("Realized P&L:", completed_state.realized_pnl_usd)
print("Remaining cost basis:", remaining_cost_basis(completed_state))
print("Holdings:", completed_state.holdings)
print("NAV, 2026-01-05 / v1:", baseline.nav_usd)
print("NAV, 2026-01-05 / v2:", revised.nav_usd)
print("MARK REVISION IMPACT:", revised.nav_usd - baseline.nav_usd)
print("NAV, 2026-01-06 / v1:", later_mark.nav_usd)

QUANTITATIVE RESEARCH — STATE INTEGRITY REPORT
State version: 1
Pinned execution archive: market-v1
Cash: 82074.00
Realized P&L: 78.60
Remaining cost basis: 18004.60
Holdings: (('AAPL', Holding(shares=80, avg_cost_usd=Decimal('100.02'))), ('MSFT', Holding(shares=50, avg_cost_usd=Decimal('200.06'))))
NAV, 2026-01-05 / v1: 100294.00
NAV, 2026-01-05 / v2: 100344.00
MARK REVISION IMPACT: 50.00
NAV, 2026-01-06 / v1: 100804.00


### **Analytical Interpretation**

**1. Cash and average cost** — The two purchases use $20,005 of cash including commissions; the partial sale generates net cash proceeds of $2,079.

**2. Realized profit** — Selling 20 AAPL shares at $104 against an average basis of $100.02, less $1 commission, produces $78.60 realized P&L.

**3. NAV reconciliation** — AAPL market value is $8,320 and MSFT is $9,900, so $82,074 cash plus $18,220 securities equals $100,294 NAV.

**4. Revision sensitivity** — Repricing 50 MSFT shares from $198 to $199 raises NAV by $50, with no change to cash, shares, or realized P&L.

**5. Temporal isolation** — At the Jan 5 reporting date, the archive cannot use Jan 6 observations. A Jan 6 valuation deliberately uses the later market date.

**6. Assignment-to-provenance link** — The committed ledger points to an immutable state and a named archive version; rebinding an analysis to a revised archive does not rewrite historical fills.

## 2.11.17 Deterministic Replay, Provenance, and Identity Audit

### Theoretical Foundations

A stored research result should be reproducible from **the same initial state, ordered fills, accounting policy, market archive version, and batch boundaries**. A canonical fingerprint identifies the serialized content of a selected state; a content hash is **not** a signature, trusted provenance, or authentication mechanism. Replaying commits with the same batch boundaries preserves the semantic version counter.

An important distinction is that **equality of two immutable ledger states does not mean object identity**. Independent runs should normally have different Python object identities and equal economic state. The market archive fingerprint and valuation date are separate dependencies of a NAV report; ledger state alone does not determine marked-to-market NAV.

In [17]:
import hashlib
import json

def ledger_fingerprint(state: LedgerState) -> str:
    """Hash a canonical representation of public ledger state."""
    payload = {
        "version": state.version,
        "initial_cash_usd": str(state.initial_cash_usd),
        "cash_usd": str(state.cash_usd),
        "realized_pnl_usd": str(state.realized_pnl_usd),
        "holdings": [
            (ticker, holding.shares, str(holding.avg_cost_usd))
            for ticker, holding in state.holdings
        ],
        "fill_history": [
            (fill.fill_id, fill.day.isoformat(), fill.side, fill.ticker,
             fill.shares, str(fill.execution_price_usd), str(fill.fee_usd))
            for fill in state.fills
        ],
        "processed_ids": sorted(state.processed_ids),
        "archive_version": state.archive_version,
    }
    wire = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(wire.encode("utf-8")).hexdigest()

def replay_ledger(
    *,
    starting_cash: Decimal,
    policy: RiskPolicy,
    archive: MarketArchive,
    batches: tuple[tuple[tuple[Fill, ...], date], ...],
) -> LedgerState:
    fresh = ResearchLedger(cash_usd=starting_cash, policy=policy)
    for batch, evaluation_day in batches:
        fresh.commit_batch(
            batch,
            archive=archive,
            expected_version=fresh.snapshot().version,
            evaluation_day=evaluation_day,
        )
    return fresh.snapshot()

replayed = replay_ledger(
    starting_cash=D("100000"),
    policy=policy,
    archive=archive_v1,
    batches=((parsed_fills, date(2026, 1, 5)),),
)
assert replayed == completed_state
assert replayed is not completed_state
assert ledger_fingerprint(replayed) == ledger_fingerprint(completed_state)
assert archive_v1.fingerprint() != archive_v2.fingerprint()

report_manifest = {
    "ledger_version": completed_state.version,
    "ledger_sha256": ledger_fingerprint(completed_state),
    "market_version": archive_v1.version,
    "market_sha256": archive_v1.fingerprint(),
    "report_asof": "2026-01-05",
    "valuation_usd": str(baseline.nav_usd),
    "policy_max_position_weight": str(policy.max_position_weight),
}
print("REPRODUCIBLE RESEARCH MANIFEST")
print(json.dumps(report_manifest, indent=2))
print("Independent replay matches:", replayed == completed_state)

REPRODUCIBLE RESEARCH MANIFEST
{
  "ledger_version": 1,
  "ledger_sha256": "234411c06b593525b20cd318566c5259cdb6010aa30629eb1d7ea2d8d5f94827",
  "market_version": "market-v1",
  "market_sha256": "1ea997ca2762ee620b7f727de855fbfdc24ab7b473aa74cc9fbcdee7ff0003b0",
  "report_asof": "2026-01-05",
  "valuation_usd": "100294.00",
  "policy_max_position_weight": "0.20"
}
Independent replay matches: True


### **Analytical Interpretation**

**1. Reconstruction from inputs** — Replaying the same immutable records through the same policy, archive, and batch boundaries produces an equal ledger state.

**2. Equality vs identity** — The replayed state is a separately constructed object, so equality does not imply shared identity.

**3. Content fingerprinting** — The deterministic SHA-256 digest tracks a specified canonical representation; it cannot prove data correctness or origin.

**4. Complete provenance** — A marked-to-market report depends on both ledger data and a particular market archive, date, and risk methodology.

**5. Research discipline** — Store sufficient configuration and data-version metadata to reproduce calculations after the active Python process ends.

## 2.11.18 Automated Verification: Language Semantics, Finance, and Failure Paths

### Verification Plan

A final capstone must be evaluated by **negative tests and invariant tests**, not only by a successful reference scenario. This suite covers:

1. **Python semantics** — aliasing, copy independence, immutable additions, name and context behavior.
2. **Market archive** — as-of correctness, revision preconditions, duplicate records, data version independence, fingerprints.
3. **Type and input contracts** — invalid dates, Boolean share counts, nonfinite Decimals, negative fees, inconsistent policy thresholds.
4. **Financial transitions** — buy/sell cost bases, commissions, realized P&L, position removal, insufficient funds, overselling, chronology.
5. **Risk constraints** — per-position, gross exposure, cash reserve, and maximum order notional.
6. **Batch publication** — stale version conflicts, partial-batch rollback, duplicate IDs, immutable prior snapshots, market version pinning.
7. **Reproducibility** — deterministic replay, content fingerprints, sensitivity to revised quotes, and source preservation.
8. **Generated cases** — deterministic randomized trade sequences and repeated accounting identities.

A passing test suite establishes correctness **for the tested contracts and scenarios**, not full correctness for all possible external financial systems. The tests use `unittest` and do not require third-party packages.

In [18]:
import unittest
from concurrent.futures import ThreadPoolExecutor
from dataclasses import FrozenInstanceError
from datetime import date
from decimal import Decimal as D
from threading import Barrier
from random import Random

TEST_DAY = date(2026, 1, 5)

def make_ledger(
    custom_policy: RiskPolicy | None = None,
    cash: Decimal = D("100000"),
) -> ResearchLedger:
    return ResearchLedger(cash_usd=cash, policy=custom_policy or policy)

def mk_fill(
    fill_id: str,
    *,
    day: date = TEST_DAY,
    side: TradeSide = "BUY",
    ticker: str = "AAPL",
    shares: int = 1,
    price: str = "104",
    fee: str = "0",
) -> Fill:
    return Fill(fill_id, day, side, ticker, shares, D(price), D(fee))

class AssignmentSemanticsTests(unittest.TestCase):
    def test_alias_identity(self):
        obj = [1]; alias = obj
        self.assertIs(obj, alias)

    def test_rebinding_preserves_old_object(self):
        original = [1]; alias = original; alias = [2]
        self.assertEqual(original, [1]); self.assertIsNot(alias, original)

    def test_shallow_nested_aliasing(self):
        original = {"x": [1]}; shallow = copy(original)
        shallow["x"].append(2)
        self.assertEqual(original["x"], [1, 2])

    def test_deep_copy_independence(self):
        original = {"x": [1]}; clone = deepcopy(original)
        clone["x"].append(2)
        self.assertEqual(original["x"], [1])

    def test_mappingproxy_is_live(self):
        original = {"x": 1}; view = MappingProxyType(original)
        original["x"] = 2
        self.assertEqual(view["x"], 2)

    def test_mappingproxy_rejects_item_assignment(self):
        view = MappingProxyType({"x": 1})
        with self.assertRaises(TypeError):
            view["x"] = 3

    def test_inplace_returns_same_mutable_object(self):
        x = MutablePnL(D("5")); alias = x; x += ImmutablePnL(D("2"))
        self.assertIs(x, alias); self.assertEqual(x.usd, D("7"))

    def test_pure_addition_keeps_old_object(self):
        x = ImmutablePnL(D("5")); y = x + ImmutablePnL(D("2"))
        self.assertEqual(x.usd, D("5")); self.assertEqual(y.usd, D("7"))

    def test_newtype_runtime_identity(self):
        identifier = FillId("F001")
        self.assertIsInstance(identifier, str)

    def test_annotations_do_not_validate(self):
        x: Decimal = "not-a-decimal"
        self.assertIsInstance(x, str)

    def test_walrus_preserves_zero(self):
        items = {"AAPL": 0}
        self.assertIsNotNone((result := items.get("AAPL")))
        self.assertEqual(result, 0)

    def test_separate_research_contexts(self):
        var = ContextVar("test-context", default="none")
        a = Context(); b = Context()
        self.assertEqual(a.run(lambda: (var.set("A"), var.get())[1]), "A")
        self.assertEqual(b.run(lambda: (var.set("B"), var.get())[1]), "B")
        self.assertEqual(var.get(), "none")

    def test_ast_classification(self):
        parsed = ast.parse("x = 1\nbook.price = 2\na['x'] = 3")
        targets = [type(node.targets[0]).__name__ for node in parsed.body]
        self.assertEqual(targets, ["Name", "Attribute", "Subscript"])

class MarketArchiveTests(unittest.TestCase):
    def setUp(self):
        self.archive = make_market_archive()

    def test_point_in_time_before_next_day(self):
        mark = self.archive.asof(date(2026, 1, 3))
        self.assertEqual(mark.price("AAPL"), D("100"))

    def test_point_in_time_latest_on_day(self):
        self.assertEqual(self.archive.asof(TEST_DAY).price("AAPL"), D("104"))

    def test_point_in_time_excludes_future(self):
        self.assertEqual(self.archive.asof(TEST_DAY).price("MSFT"), D("198"))

    def test_source_date_retained(self):
        self.assertEqual(
            self.archive.asof(date(2026, 1, 4)).source_day("JPM"),
            date(2026, 1, 2),
        )

    def test_pre_history_asof_rejected(self):
        with self.assertRaises(ValueError):
            self.archive.asof(date(2025, 12, 31))

    def test_missing_ticker_rejected(self):
        with self.assertRaises(KeyError):
            self.archive.asof(TEST_DAY).price("GOOGL")

    def test_original_unchanged_after_revision(self):
        updated = self.archive.revise(day=TEST_DAY, ticker="MSFT", expected_old_price=D("198"), revised_price=D("199"), new_version="v2")
        self.assertEqual(self.archive.asof(TEST_DAY).price("MSFT"), D("198"))
        self.assertEqual(updated.asof(TEST_DAY).price("MSFT"), D("199"))

    def test_revision_requires_matching_old_price(self):
        with self.assertRaises(ValueError):
            self.archive.revise(day=TEST_DAY, ticker="MSFT", expected_old_price=D("197"), revised_price=D("199"), new_version="v2")

    def test_revision_requires_new_version(self):
        with self.assertRaises(ValueError):
            self.archive.revise(day=TEST_DAY, ticker="MSFT", expected_old_price=D("198"), revised_price=D("199"), new_version=self.archive.version)

    def test_revision_missing_target_rejected(self):
        with self.assertRaises(KeyError):
            self.archive.revise(day=TEST_DAY, ticker="GOOGL", expected_old_price=D("100"), revised_price=D("101"), new_version="v2")

    def test_nonpositive_revision_rejected(self):
        with self.assertRaises(ValueError):
            self.archive.revise(day=TEST_DAY, ticker="MSFT", expected_old_price=D("198"), revised_price=D("0"), new_version="v2")

    def test_canonical_sort_fingerprint(self):
        reversed_archive = MarketArchive("market-v1", tuple(reversed(self.archive.quotes)))
        self.assertEqual(self.archive.fingerprint(), reversed_archive.fingerprint())

    def test_revision_changes_fingerprint(self):
        updated = self.archive.revise(day=TEST_DAY, ticker="MSFT", expected_old_price=D("198"), revised_price=D("199"), new_version="v2")
        self.assertNotEqual(self.archive.fingerprint(), updated.fingerprint())

    def test_duplicate_quote_rejected(self):
        with self.assertRaises(ValueError):
            MarketArchive("invalid", self.archive.quotes + (self.archive.quotes[0],))

    def test_empty_archive_rejected(self):
        with self.assertRaises(ValueError):
            MarketArchive("empty", ())

    def test_invalid_quote_float(self):
        with self.assertRaises(TypeError):
            Quote(TEST_DAY, "AAPL", 100.0)

    def test_invalid_quote_nonfinite(self):
        with self.assertRaises(TypeError):
            Quote(TEST_DAY, "AAPL", D("NaN"))

    def test_invalid_quote_negative(self):
        with self.assertRaises(ValueError):
            Quote(TEST_DAY, "AAPL", D("-1"))

    def test_invalid_quote_day(self):
        with self.assertRaises(TypeError):
            Quote("2026-01-05", "AAPL", D("100"))

class FinancialContractTests(unittest.TestCase):
    def test_fill_parsing(self):
        self.assertEqual(parse_fill(raw_trade_rows[0]), parsed_fills[0])

    def test_missing_fill_metadata(self):
        with self.assertRaises(ValueError):
            parse_fill(raw_trade_rows[0][:7])

    def test_nonstring_price_source_rejected(self):
        row = list(raw_trade_rows[0]); row[5] = 100.0
        with self.assertRaises(TypeError):
            parse_fill(tuple(row))

    def test_fill_rejects_boolean_quantity(self):
        with self.assertRaises(ValueError):
            Fill("X", TEST_DAY, "BUY", "AAPL", True, D("10"), D("0"))

    def test_fill_rejects_zero_quantity(self):
        with self.assertRaises(ValueError):
            mk_fill("X", shares=0)

    def test_fill_rejects_float_price(self):
        with self.assertRaises(TypeError):
            Fill("X", TEST_DAY, "BUY", "AAPL", 1, 10.0, D("0"))

    def test_fill_rejects_negative_fee(self):
        with self.assertRaises(ValueError):
            mk_fill("X", fee="-1")

    def test_fill_rejects_nan_fee(self):
        with self.assertRaises(TypeError):
            mk_fill("X", fee="NaN")

    def test_fill_rejects_invalid_side(self):
        with self.assertRaises(ValueError):
            mk_fill("X", side="HOLD")

    def test_fill_rejects_empty_identifier(self):
        with self.assertRaises(ValueError):
            mk_fill("")

    def test_fill_rejects_string_day(self):
        with self.assertRaises(TypeError):
            Fill("X", "2026-01-05", "BUY", "AAPL", 1, D("1"), D("0"))

    def test_fill_is_frozen(self):
        fill = mk_fill("X")
        with self.assertRaises(FrozenInstanceError):
            fill.shares = 10

    def test_risk_policy_rejects_bad_weight(self):
        with self.assertRaises(ValueError):
            RiskPolicy(D("1.1"), D("1.1"), D("100"))

    def test_risk_policy_rejects_inconsistent_weights(self):
        with self.assertRaises(ValueError):
            RiskPolicy(D("0.4"), D("0.3"), D("100"))

    def test_risk_policy_rejects_negative_cash_reserve(self):
        with self.assertRaises(ValueError):
            RiskPolicy(D("0.2"), D("0.3"), D("100"), D("-1"))

    def test_initial_cash_rejects_float(self):
        with self.assertRaises(TypeError):
            initial_state(100.0)

    def test_negative_cash_rejected(self):
        with self.assertRaises(ValueError):
            initial_state(D("-1"))

    def test_holding_rejects_zero_shares(self):
        with self.assertRaises(ValueError):
            Holding(0, D("100"))

    def test_processed_id_invariant(self):
        state = initial_state(D("10"))
        with self.assertRaises(ValueError):
            replace(state, processed_ids=frozenset({"unexpected"}))

    def test_holdings_sorted_invariant(self):
        state = initial_state(D("100"))
        with self.assertRaises(ValueError):
            replace(state, holdings=(("MSFT", Holding(1, D("1"))), ("AAPL", Holding(1, D("1")))))

class FinancialTransitionTests(unittest.TestCase):
    def setUp(self):
        self.market = make_market_archive()
        self.base = initial_state(D("100000"))

    def test_single_purchase_cash_and_basis(self):
        state = transition_fill(self.base, parsed_fills[0], self.market, policy)
        self.assertEqual(state.cash_usd, D("89998"))
        self.assertEqual(dict(state.holdings)["AAPL"], Holding(100, D("100.02")))

    def test_single_sale_realized_pnl(self):
        state = self.base
        for fill in parsed_fills:
            state = transition_fill(state, fill, self.market, policy)
        self.assertEqual(state.realized_pnl_usd, D("78.60"))

    def test_final_cash(self):
        state = self.base
        for fill in parsed_fills:
            state = transition_fill(state, fill, self.market, policy)
        self.assertEqual(state.cash_usd, D("82074"))

    def test_average_cost_after_partial_sale(self):
        state = self.base
        for fill in parsed_fills:
            state = transition_fill(state, fill, self.market, policy)
        self.assertEqual(dict(state.holdings)["AAPL"].avg_cost_usd, D("100.02"))

    def test_accounting_reconciliation(self):
        state = self.base
        for fill in parsed_fills:
            state = transition_fill(state, fill, self.market, policy)
        self.assertEqual(state.cash_usd + remaining_cost_basis(state), self.base.initial_cash_usd + state.realized_pnl_usd)

    def test_sale_cannot_exceed_holdings(self):
        with self.assertRaises(ValueError):
            transition_fill(self.base, mk_fill("S", side="SELL", shares=1), self.market, policy)

    def test_duplicate_fill_rejected(self):
        state = transition_fill(self.base, parsed_fills[0], self.market, policy)
        with self.assertRaises(ValueError):
            transition_fill(state, parsed_fills[0], self.market, policy)

    def test_insufficient_cash_rejected(self):
        low = initial_state(D("10"))
        with self.assertRaises(ValueError):
            transition_fill(low, mk_fill("B", shares=1), self.market, policy)

    def test_missing_market_quote_rejected(self):
        with self.assertRaises(KeyError):
            transition_fill(self.base, mk_fill("B", ticker="GOOGL"), self.market, policy)

    def test_order_notional_limit_rejected(self):
        with self.assertRaises(ValueError):
            transition_fill(self.base, mk_fill("B", shares=250), self.market, policy)

    def test_sale_fee_exceeds_proceeds_rejected(self):
        state = transition_fill(self.base, parsed_fills[0], self.market, policy)
        with self.assertRaises(ValueError):
            transition_fill(state, mk_fill("S", side="SELL", shares=1, price="1", fee="2"), self.market, policy)

    def test_fully_closed_position_removed(self):
        state = transition_fill(self.base, parsed_fills[0], self.market, policy)
        state = transition_fill(state, mk_fill("S", side="SELL", shares=100, price="104"), self.market, policy)
        self.assertEqual(state.holdings, ())

    def test_nonchronological_fill_rejected(self):
        state = transition_fill(self.base, mk_fill("B"), self.market, policy)
        earlier = mk_fill("E", day=date(2026, 1, 2), price="100")
        with self.assertRaises(ValueError):
            transition_fill(state, earlier, self.market, policy)

    def test_market_version_pinning_rejected(self):
        state = transition_fill(self.base, mk_fill("B"), self.market, policy)
        revised_archive = self.market.revise(day=TEST_DAY, ticker="MSFT", expected_old_price=D("198"), revised_price=D("199"), new_version="v2")
        with self.assertRaises(ValueError):
            transition_fill(state, mk_fill("B2"), revised_archive, policy)

    def test_transition_preserves_original_state(self):
        old = self.base
        newer = transition_fill(old, parsed_fills[0], self.market, policy)
        self.assertEqual(old.cash_usd, D("100000"))
        self.assertNotEqual(newer.cash_usd, old.cash_usd)

    def test_realized_and_unrealized_pnl_identity(self):
        state = self.base
        for fill in parsed_fills:
            state = transition_fill(state, fill, self.market, policy)
        valuation = mark_portfolio(state, self.market.asof(TEST_DAY))
        self.assertEqual(valuation.nav_usd - state.initial_cash_usd, valuation.realized_pnl_usd + valuation.unrealized_pnl_usd)

    def test_marked_nav_baseline(self):
        state = self.base
        for fill in parsed_fills:
            state = transition_fill(state, fill, self.market, policy)
        self.assertEqual(mark_portfolio(state, self.market.asof(TEST_DAY)).nav_usd, D("100294"))

    def test_mark_asof_missing_asset_rejected(self):
        state = transition_fill(self.base, parsed_fills[0], self.market, policy)
        incomplete = MarketMark("x", TEST_DAY, (Quote(TEST_DAY, "MSFT", D("200")),))
        with self.assertRaises(KeyError):
            mark_portfolio(state, incomplete)

    def test_risk_max_position_rejected(self):
        state = self.base
        for fill in parsed_fills:
            state = transition_fill(state, fill, self.market, policy)
        concentrated = transition_fill(state, mk_fill("B4", shares=170), self.market, policy)
        with self.assertRaises(ValueError):
            enforce_risk(concentrated, self.market.asof(TEST_DAY), policy)

    def test_risk_gross_rejected(self):
        modified_policy = RiskPolicy(D("0.10"), D("0.15"), D("20000"))
        state = self.base
        for fill in parsed_fills:
            state = transition_fill(state, fill, self.market, modified_policy)
        with self.assertRaises(ValueError):
            enforce_risk(state, self.market.asof(TEST_DAY), modified_policy)

class LedgerCommitTests(unittest.TestCase):
    def setUp(self):
        self.market = make_market_archive()
        self.ledger = make_ledger()

    def commit_reference(self):
        return self.ledger.commit_batch(parsed_fills, archive=self.market, expected_version=0, evaluation_day=TEST_DAY)

    def test_reference_state(self):
        self.assertEqual(self.commit_reference().cash_usd, D("82074"))

    def test_success_increments_version_once(self):
        self.assertEqual(self.commit_reference().version, 1)

    def test_snapshot_identity_before_and_after(self):
        before = self.ledger.snapshot()
        after = self.commit_reference()
        self.assertIsNot(before, after)
        self.assertEqual(before.cash_usd, D("100000"))

    def test_old_snapshot_unchanged(self):
        before = self.ledger.snapshot()
        self.commit_reference()
        self.assertEqual(before, initial_state(D("100000")))

    def test_invalid_later_trade_rolls_back(self):
        before = self.ledger.snapshot()
        with self.assertRaises(ValueError):
            self.ledger.commit_batch((parsed_fills[0], mk_fill("S", side="SELL", shares=500)), archive=self.market, expected_version=0, evaluation_day=TEST_DAY)
        self.assertEqual(self.ledger.snapshot(), before)

    def test_duplicate_within_batch_rolls_back(self):
        with self.assertRaises(ValueError):
            self.ledger.commit_batch((parsed_fills[0], parsed_fills[0]), archive=self.market, expected_version=0, evaluation_day=TEST_DAY)
        self.assertEqual(self.ledger.snapshot().version, 0)

    def test_duplicate_committed_id_rejected(self):
        before = self.commit_reference()
        with self.assertRaises(ValueError):
            self.ledger.commit_batch((parsed_fills[0],), archive=self.market, expected_version=1, evaluation_day=TEST_DAY)
        self.assertEqual(before, self.ledger.snapshot())

    def test_stale_version_rejected(self):
        self.commit_reference()
        with self.assertRaises(ValueError):
            self.ledger.commit_batch((mk_fill("B9"),), archive=self.market, expected_version=0, evaluation_day=TEST_DAY)

    def test_future_event_rejected(self):
        with self.assertRaises(ValueError):
            self.ledger.commit_batch((mk_fill("B9", day=date(2026, 1, 6)),), archive=self.market, expected_version=0, evaluation_day=TEST_DAY)

    def test_empty_batch_rejected(self):
        with self.assertRaises(ValueError):
            self.ledger.commit_batch((), archive=self.market, expected_version=0, evaluation_day=TEST_DAY)

    def test_list_batch_rejected(self):
        with self.assertRaises(ValueError):
            self.ledger.commit_batch([mk_fill("B9")], archive=self.market, expected_version=0, evaluation_day=TEST_DAY)

    def test_non_fill_record_rejected(self):
        with self.assertRaises(TypeError):
            self.ledger.commit_batch(("not-a-fill",), archive=self.market, expected_version=0, evaluation_day=TEST_DAY)
        self.assertEqual(self.ledger.snapshot().version, 0)

    def test_changed_archive_rejected_for_new_trades(self):
        self.commit_reference()
        updated = self.market.revise(day=TEST_DAY, ticker="MSFT", expected_old_price=D("198"), revised_price=D("199"), new_version="v2")
        with self.assertRaises(ValueError):
            self.ledger.commit_batch((mk_fill("B9"),), archive=updated, expected_version=1, evaluation_day=TEST_DAY)

    def test_revised_archive_reprices_without_mutation(self):
        state = self.commit_reference()
        updated = self.market.revise(day=TEST_DAY, ticker="MSFT", expected_old_price=D("198"), revised_price=D("199"), new_version="v2")
        before = mark_portfolio(state, self.market.asof(TEST_DAY))
        after = mark_portfolio(state, updated.asof(TEST_DAY))
        self.assertEqual(after.nav_usd - before.nav_usd, D("50"))
        self.assertEqual(state, self.ledger.snapshot())

    def test_replay_same_state(self):
        state = self.commit_reference()
        replayed = replay_ledger(starting_cash=D("100000"), policy=policy, archive=self.market, batches=((parsed_fills, TEST_DAY),))
        self.assertEqual(state, replayed)
        self.assertIsNot(state, replayed)

    def test_replay_fingerprint(self):
        state = self.commit_reference()
        replayed = replay_ledger(starting_cash=D("100000"), policy=policy, archive=self.market, batches=((parsed_fills, TEST_DAY),))
        self.assertEqual(ledger_fingerprint(state), ledger_fingerprint(replayed))

    def test_position_limit_failure_rolls_back(self):
        self.commit_reference()
        before = self.ledger.snapshot()
        with self.assertRaises(ValueError):
            self.ledger.commit_batch((mk_fill("B9", shares=170),), archive=self.market, expected_version=1, evaluation_day=TEST_DAY)
        self.assertEqual(self.ledger.snapshot(), before)

    def test_cash_reserve_failure_rolls_back(self):
        restricted = RiskPolicy(D("1"), D("1"), D("20000"), D("99990"))
        book = make_ledger(custom_policy=restricted)
        with self.assertRaises(ValueError):
            book.commit_batch((mk_fill("B9"),), archive=self.market, expected_version=0, evaluation_day=TEST_DAY)
        self.assertEqual(book.snapshot().cash_usd, D("100000"))

    def test_concurrent_expected_version_one_winner(self):
        fresh = make_ledger()
        gate = Barrier(2)
        def worker(fill):
            gate.wait()
            try:
                fresh.commit_batch((fill,), archive=self.market, expected_version=0, evaluation_day=TEST_DAY)
                return "success"
            except ValueError as exc:
                return str(exc)
        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = [
                executor.submit(worker, mk_fill("T-A", ticker="AAPL", price="104")),
                executor.submit(worker, mk_fill("T-M", ticker="MSFT", price="198")),
            ]
            results = [future.result() for future in futures]
        self.assertEqual(results.count("success"), 1)
        self.assertEqual(results.count("Stale ledger version"), 1)
        self.assertEqual(fresh.snapshot().version, 1)

class GeneratedInvariantTests(unittest.TestCase):
    def test_120_deterministic_randomized_buys(self):
        rng = Random(20261123)
        wide = RiskPolicy(D("1"), D("1"), D("100000"))
        ledger = make_ledger(custom_policy=wide, cash=D("1000000"))
        for i in range(120):
            ticker = rng.choice(("AAPL", "MSFT", "JPM"))
            quantity = rng.randint(1, 9)
            price = {"AAPL": "104", "MSFT": "198", "JPM": "52"}[ticker]
            trade = mk_fill(f"R{i:04d}", ticker=ticker, shares=quantity, price=price, fee="0.01")
            state = ledger.commit_batch((trade,), archive=archive_v1, expected_version=ledger.snapshot().version, evaluation_day=TEST_DAY)
            assert_accounting_identity(state)
            self.assertGreaterEqual(state.cash_usd, D("0"))
        self.assertEqual(ledger.snapshot().version, 120)
        self.assertEqual(len(ledger.snapshot().processed_ids), 120)

    def test_many_repeated_replays_stable(self):
        fingerprints = set()
        for _ in range(15):
            state = replay_ledger(starting_cash=D("100000"), policy=policy, archive=archive_v1, batches=((parsed_fills, TEST_DAY),))
            fingerprints.add(ledger_fingerprint(state))
        self.assertEqual(len(fingerprints), 1)

    def test_revised_marks_never_change_committed_cash(self):
        state = replay_ledger(starting_cash=D("100000"), policy=policy, archive=archive_v1, batches=((parsed_fills, TEST_DAY),))
        old_cash = state.cash_usd
        for mark in (archive_v1.asof(TEST_DAY), archive_v2.asof(TEST_DAY), archive_v1.asof(date(2026, 1, 6))):
            mark_portfolio(state, mark)
            self.assertEqual(state.cash_usd, old_cash)

def run_full_verification() -> unittest.result.TestResult:
    loader = unittest.TestLoader()
    suite = unittest.TestSuite()
    for case in (
        AssignmentSemanticsTests,
        MarketArchiveTests,
        FinancialContractTests,
        FinancialTransitionTests,
        LedgerCommitTests,
        GeneratedInvariantTests,
    ):
        suite.addTests(loader.loadTestsFromTestCase(case))
    import sys
    result = unittest.TextTestRunner(stream=sys.stdout, verbosity=1).run(suite)
    if not result.wasSuccessful():
        raise AssertionError("Research state integrity verification failed")
    return result

result_211 = run_full_verification()
print("Verified test cases:", result_211.testsRun)

.

.

.

.

.

.

... (86 outputs left out)

.

.

----------------------------------------------------------------------
Ran 94 tests in 0.035s

OK


Verified test cases: 94


### **Analytical Interpretation**

**1. Breadth** — The tests cover source-code semantics, market versions, trade validation, cost-basis accounting, concentration rules, rollback, and concurrency preconditions.

**2. Negative paths** — A batch containing a late invalid trade must preserve the exact authoritative ledger state and version.

**3. Independent reconstruction** — Replayed transaction histories create new objects with equal values and equal content digests.

**4. Generated testing** — A seeded randomized loop checks many valid purchase transitions and the accounting identity; a seed makes those scenarios reproducible.

**5. Concurrency limits** — The two-thread test verifies one winner under equal optimistic versions for cooperating calls; it is not a proof of distributed linearizability.

**6. Precision policies** — Monetary equality depends on Decimal context and the documented numerical tolerance for nonterminating average-cost division.

## 2.11.19 Final Analytical Interpretation

### **Analytical Interpretation**

**1. Names, objects, and states.** Names point to objects, not statically typed storage slots. Rebinding a name can be harmless, while modifying a shared object can invalidate multiple experiments at once. The ledger therefore publishes a complete immutable root rather than editing shared nested dictionaries.

**2. Assignment evaluation and target dispatch.** The right-hand side is evaluated before an ordinary assignment target. An attribute or subscription target can execute arbitrary code. Writing a multi-target statement does not grant atomicity; failure can occur after earlier writes have already happened.

**3. Multiple, chained, and unpacking assignment.** Unpacking is useful for parsing financial records but consumes iterables and follows exact target rules. Chained assignment shares one evaluated object among multiple targets. Neither syntax guarantees independent containers or independent transactions.

**4. Augmented assignment.** `+=` may mutate in place; an alias may observe the modification. The notebook contrasts a deliberately pure financial addition with a deliberately mutating accumulator so that the ownership model is explicit.

**5. Annotated assignment and type hints.** Type annotations and `NewType` improve static reasoning but do not reject invalid runtime values automatically. Critical financial boundaries use explicit validation of dates, IDs, quantities, and finite `Decimal` values.

**6. Assignment expressions.** A walrus expression can avoid duplicate evaluation, but truthiness-based selection can confuse missing data with valid zero exposure. An explicit `None` sentinel resolves the distinction.

**7. Scope and closures.** A loop-captured name may be resolved late and contaminate parameter sweeps. Factories and immutable policy values prevent cross-trial capture errors; `nonlocal` supports deliberate rebinding when a stateful closure is appropriate.

**8. Object identity and aliasing.** A shallow copy may still share nested lists, and a read-only mapping proxy remains a live view. Independent market archives and immutable nested records provide a clearer point-in-time research contract.

**9. Advanced assignment pitfalls.** Errors may occur after side effects, retries may double-apply event IDs, and optimistic versions can become stale. Those concerns are addressed by immutable staged transitions, duplicate detection, and version preconditions.

**10. Point-in-time research.** For an as-of date, only quotations no later than that date enter the mark. Revisions create new archives; analytical valuation sensitivity does not alter recorded historical trades or realized P&L.

**11. Financial identities.** The cost-basis accounting identity and marked-to-market P&L identity supply independent checks on cash, inventory, and profit calculations. They complement—not replace—input validation and reproducibility requirements.

**12. Atomic publication.** A batch may involve many internal assignments while preparing temporary records. The managed state changes only after validation through one reference publication under a lock. Its guarantees are limited to the in-process API and assumed cooperative access.

**13. Replay and audit.** The same inputs, policy, archive version, and batch boundaries produce equal independent states. A deterministic fingerprint identifies canonical content, but it is not a cryptographic signature of trusted market provenance.

**14. Interpretation of the financial result.** The example ledger ends with **$82,074 cash**, **$78.60 realized P&L**, and **$100,294 NAV** at the original Jan 5 mark. Revising MSFT's Jan 5 mark by $1 changes valuation by **$50**, not realized trading profit.

**15. Professional engineering conclusion.** Reliable quantitative software requires both accurate numerical formulas and explicit object/state semantics: ownership, versioning, data availability, immutable records, transactional boundaries, accounting reconciliation, and tests for every failure path. This is the practical meaning of mastering Python's Variables and Assignment module.

## 2.11.20 Advanced Practice Exercises

### Exercise 1 — Proof of Assignment Evaluation Order

Construct a reusable tracing tool for `Name`, `Attribute`, `Subscript`, unpacking, and chained targets. Compare predictions to execution traces, test at least three failures, and explain why neither AST structure nor bytecode alone proves absence of side effects.

### Exercise 2 — Deep Aliasing Audit

Build a finite, cycle-safe object-graph inspector that distinguishes repeated references, `copy.copy`, `copy.deepcopy`, and live read-only views. Design explicit limits to avoid traversing arbitrary descriptors or triggering side effects. Show one realistic data-leakage failure and its immutable-snapshot correction.

### Exercise 3 — Type and Domain Contract Checker

Use `typing.Protocol`, `NewType`, `Literal`, overloads, runtime validators, and type-checker-friendly interfaces to distinguish dates, order IDs, security tickers, prices, shares, and risk thresholds. Create a report separating compile-time assumptions from enforced runtime invariants.

### Exercise 4 — Point-in-Time Publication History

Add `observed_at` and `published_at` timestamps to the market archive. Reject quotes published after a research decision time even if the observation date is earlier. Test out-of-order vendor messages, corrections, multiple time zones, and unsupported calendar dates.

### Exercise 5 — Portfolio Accounting under Fractional Cost Basis

Investigate nonterminating average-cost representations using `Decimal`. Compare quantization conventions, remainder-carrying cost basis, and exact rational arithmetic. Verify reconciliation after repeated partial sales, including end-of-position residual handling.

### Exercise 6 — Transaction Isolation and Lost Updates

Construct a controlled multi-threaded experiment with simultaneous expected-version writes. Demonstrate one successful commit and one stale-version rejection; compare this with a deliberately unsafe read-modify-write implementation. Explain why the GIL is not a transaction policy.

### Exercise 7 — Event Replay and Audit-Signed Manifests

Store an append-only ordered transaction record and a canonical ledger manifest. Implement restart/replay from a durable file, checksum validation, and explicit failure recovery. Differentiate integrity hashes from authenticated digital signatures.

### Exercise 8 — Multi-Archive Research Experiments

Run two independent backtests against different immutable market archive versions. Hold fills and policy fixed. Attribute the differences in NAV to individual revised securities, and explicitly separate revision sensitivity from strategy performance.

### Exercise 9 — Stateful Parameter Search with Closures

Run a grid of policy limits using factory-created immutable configurations. Introduce a late-binding defect intentionally, prove the incorrect behavior, fix it, and verify that scenario results are independent of execution order.

### Exercise 10 — Static Assignment Risk Linter

Develop an AST analyzer that flags `+=` on subscription targets, function calls in assignment targets, chained assignment to mutable literals, and assignments to likely global state. Include false positives, false negatives, and source-line diagnostics.

### Exercise 11 — Risk Engine Extensions

Introduce industry exposures, security-level position limits, liquidity haircuts, fee schedules, maximum daily turnover, and mark-age limits. Enforce risk after each staged event and on the final reporting date without mutating committed state.

### Exercise 12 — Differential and Metamorphic Testing

Write an independent slow reference engine. Generate deterministic valid and invalid fill batches, and compare cash, cost basis, P&L, holdings, rollback, versioning, and fingerprints. Report any differences rather than accepting both implementations as correct by construction.

### Exercise Submission Standard

Each exercise must contain **theoretical assumptions, executable code, expected results, `### **Analytical Interpretation**`, failure-path tests, and limitations**. Tests must run from a clean notebook kernel. Avoid hardcoded implementation-specific object identifiers.

## Final Expert Challenge — Research-Grade Event-Sourced Portfolio Control Plane

### Objective

Extend the capstone into a **research-grade, event-sourced portfolio state engine** with point-in-time market provenance, reproducible risk evaluation, deterministic replay, and explicit recovery from failed updates. All input records and policies must be versioned, and each committed state must be reconstructible without hidden Jupyter global variables.

### Required Workstreams

| Workstream | Deliverables |
|---|---|
| Language-semantics audit | AST assignment classification, aliased-object review, execution-order counterexamples |
| Domain and type system | Strict money/quantity/date/identifier types, explicit validation, static type-checking configuration |
| Market data provenance | Observation and publication timestamps, data revision graph, immutable as-of snapshots |
| Execution ledger | Unique fill IDs, long/short policy decisions, average and/or lot-based cost accounting |
| Risk management | Per-security, gross/net, cash, liquidity, sector, turnover, and data-freshness constraints |
| State publication | Independent staging, optimistic versions, audited commit, versioned snapshots |
| Durability | Append-only write-ahead events, crash recovery, durable commit markers, file/database transaction strategy |
| Replay and reconciliation | Independent reference model, exact input manifests, accounting invariants |
| Concurrency | Competing commits, consistent reads, synchronization and conflict-handling policy |
| Testing | Failure injection, deterministic generated cases, numerical stress testing, performance profiling |
| Documentation | Architecture, domain assumptions, API contracts, threat model, known limitations |

### Expert Evaluation Rubric

| Dimension | Weight |
|---|---:|
| Python assignment and object-model correctness | 20% |
| Financial accounting and numerical precision | 20% |
| Point-in-time provenance and no-look-ahead controls | 15% |
| State isolation, conflict handling, and recoverability | 20% |
| Verification, negative tests, and independent reconciliation | 15% |
| Documentation and reproducible research packaging | 10% |

### Completion Criteria

1. Every accepted event has an auditable state version and immutable provenance record.
2. A failed batch leaves no externally visible partial financial state.
3. All accounting invariants are checked with documented precision/rounding rules.
4. All valuation inputs are demonstrably eligible at the requested research time.
5. Deterministic replay reproduces the recorded state without dependence on cell execution order.
6. Mutating one research scenario cannot change another scenario's saved data, policy, or result.
7. A documented durability mechanism addresses process failure; in-memory locks alone do not satisfy this requirement.
8. The solution distinguishes Python language guarantees from application-level transactional properties.

### Recommended References

- [Python Language Reference — Simple Statements](https://docs.python.org/3/reference/simple_stmts.html)
- [Python Language Reference — Expressions and Evaluation Order](https://docs.python.org/3/reference/expressions.html)
- [Python Data Model](https://docs.python.org/3/reference/datamodel.html)
- [Python `dataclasses`](https://docs.python.org/3/library/dataclasses.html)
- [Python `typing`](https://docs.python.org/3/library/typing.html)
- [Python `contextvars`](https://docs.python.org/3/library/contextvars.html)
- [Python `decimal`](https://docs.python.org/3/library/decimal.html)
- [Python `ast` and `symtable`](https://docs.python.org/3/library/ast.html)

**Final deliverable:** a standalone, documented, tested Python package plus a clean, reproducible Jupyter report. The package should expose a clear API and never require executing notebook cells out of order to obtain correct results.